
<div dir="rtl" align="right">

# سؤال ۱: تعیین واحد تحلیل (Unit of Analysis) و ساختار داده

## سؤال

با توجه به هدف پروژه Online Retail، واحد تحلیل مناسب برای آماده‌سازی داده چیست؟

مشخص کنید هر ردیف از دیتاست خام نشان‌دهنده‌ی چه چیزی است و برای مراحل بعدی پروژه به چه سطحی از داده نیاز داریم.

موارد زیر را بررسی کنید:

1. آیا هر ردیف نشان‌دهنده‌ی یک مشتری، یک فاکتور یا یک قلم کالای ثبت‌شده در فاکتور است؟
2. تفاوت سطح Transaction-Level و Customer-Level چیست؟
3. چرا برای محاسبه‌ی ویژگی‌های LRFM و انجام Customer Segmentation باید داده‌ها در سطح مشتری تجمیع شوند؟
4. چرا برای Classification مبتنی بر رفتار زمانی مشتری، باید امکان ساخت چند نمونه از یک مشتری در ReferenceDateهای متفاوت را داشته باشیم؟
5. ساختار داده‌ی موردنیاز برای هر یک از دو مسیر Clustering و Classification چیست؟

## چرایی و دلیل

دیتاست خام Online Retail در سطح ردیف‌های تراکنش محصول ثبت شده است؛ بنابراین هر ردیف الزاماً نماینده‌ی یک مشتری مستقل نیست.

یک مشتری می‌تواند در چندین فاکتور و هر فاکتور در چندین ردیف محصول حضور داشته باشد.

اگر بدون توجه به سطح تحلیل، مستقیماً روی ردیف‌های خام مدل‌سازی انجام دهیم، ممکن است یک مشتری چندین بار به‌عنوان مشاهدات مستقل در نظر گرفته شود و رفتار واقعی مشتری به‌درستی نمایش داده نشود.

## هدف

تعیین سطح صحیح تحلیل و طراحی ساختار داده‌ی متناسب با دو مسئله‌ی اصلی پروژه:

- Customer Segmentation با الگوریتم K-Means
- Churn Classification با الگوریتم XGBoost

## منطق تصمیم‌گیری

برای Clustering، هر مشتری در یک ReferenceDate مشخص باید یک مشاهده داشته باشد و ویژگی‌های LRFM و رفتاری او از تراکنش‌های معتبر استخراج شوند.

برای Classification، هر مشاهده نماینده‌ی رفتار یک مشتری تا یک ReferenceDate مشخص است و برچسب Churn با استفاده از رفتار آینده‌ی همان مشتری در Prediction Horizon ساخته می‌شود.

بنابراین ممکن است یک مشتری در تاریخ‌های مرجع متفاوت، چند نمونه‌ی تاریخی داشته باشد؛ اما این نمونه‌ها نباید به‌صورت تصادفی بین Train و Test تقسیم شوند.

## فایده

- جلوگیری از تحلیل اشتباه در سطح ردیف تراکنش
- فراهم‌کردن بستر صحیح محاسبه‌ی LRFM
- جلوگیری از تکرار غیرضروری مشتریان در Clustering
- امکان ساخت نمونه‌های زمانی برای Classification
- کاهش خطر Data Leakage
- ایجاد ساختار داده‌ی صحیح برای دو مسیر مستقل مدل‌سازی

</div>


---

In [3]:
import pandas as pd


df = pd.read_csv("../Online Retail.csv")

df.head(10)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/10 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/10 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/10 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/10 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/10 8:26,3.39,17850.0,United Kingdom
5,536365,22752,SET 7 BABUSHKA NESTING BOXES,2,12/1/10 8:26,7.65,17850.0,United Kingdom
6,536365,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,12/1/10 8:26,4.25,17850.0,United Kingdom
7,536366,22633,HAND WARMER UNION JACK,6,12/1/10 8:28,1.85,17850.0,United Kingdom
8,536366,22632,HAND WARMER RED POLKA DOT,6,12/1/10 8:28,1.85,17850.0,United Kingdom
9,536367,84879,ASSORTED COLOUR BIRD ORNAMENT,32,12/1/10 8:34,1.69,13047.0,United Kingdom


In [4]:
# Check dataset dimensions
print("Dataset Shape:", df.shape)

# Check column names
print("\nColumns:")
print(df.columns.tolist())

# Check data types
print("\nData Types:")
print(df.dtypes)

Dataset Shape: (541909, 8)

Columns:
['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']

Data Types:
InvoiceNo          str
StockCode          str
Description        str
Quantity         int64
InvoiceDate        str
UnitPrice      float64
CustomerID     float64
Country            str
dtype: object


In [5]:
# Count unique customers and invoices
print(
    "Unique Customers:",
    df["CustomerID"].nunique()
)

print(
    "Unique Invoices:",
    df["InvoiceNo"].nunique()
)

print(
    "Total Transaction Lines:",
    len(df)
)

Unique Customers: 4372
Unique Invoices: 25900
Total Transaction Lines: 541909



<div dir="rtl" align="right">

### نتایج اجرا و تحلیل سؤال اول: بررسی ساختار اولیه دیتاست

**تحلیل نتایج:**

دیتاست در سطح سطر تراکنش قرار دارد؛ یعنی هر ردیف نشان‌دهنده یک سطر ثبت‌شده از کالا در یک فاکتور است و الزاماً نماینده یک مشتری یا یک فاکتور کامل نیست.

همچنین نوع داده `CustomerID` به‌صورت `float64` ثبت شده است؛ درحالی‌که این ستون از نظر مفهومی یک شناسه است و نباید به‌عنوان متغیر عددی پیوسته تحلیل شود. وجود مقادیر گمشده می‌تواند یکی از دلایل انتخاب این نوع داده باشد.

ستون `InvoiceDate` نیز از نوع رشته است و برای انجام محاسبات زمانی باید در مراحل بعدی به نوع تاریخ و زمان تبدیل شود.

**تصمیم نهایی:**

در این مرحله هیچ ردیفی حذف یا تجمیع نمی‌شود و نوع داده‌ها نیز هنوز تغییر نمی‌کند.

ساختار داده برای ادامه پروژه به سه سطح تقسیم می‌شود:

1. داده خام در سطح سطر تراکنش.
2. داده مشتری‌محور برای الگوریتم K-Means و بخش‌بندی مشتریان.
3. داده مشتری در تاریخ مرجع (`Customer × ReferenceDate`) برای ساخت برچسب Churn و آموزش XGBoost.

**نتیجه:**

ساختار اولیه دیتاست شناسایی شد و مشخص گردید که برای اجرای دو مسئله Clustering و Classification به دو ساختار داده متفاوت در مراحل بعدی نیاز داریم.

</div>


---


<div dir="rtl" align="right">

### سؤال دوم: مقادیر گمشده و رکوردهای تکراری چه وضعیتی دارند؟

**چرایی و دلیل:**

وجود مقادیر گمشده و رکوردهای تکراری می‌تواند باعث ایجاد خطا در محاسبات و تحلیل‌های بعدی شود. برای مثال، نبودن `CustomerID` مانع انتساب یک تراکنش به مشتری مشخص می‌شود و رکوردهای تکراری ممکن است باعث شوند مقدار خرید یا تعداد تراکنش‌ها بیشتر از مقدار واقعی محاسبه شود.

بااین‌حال، هر مقدار گمشده یا هر سطر مشابهی الزاماً به معنی داده نامعتبر نیست. بنابراین قبل از تصمیم‌گیری درباره حذف، جایگزینی یا نگهداری داده‌ها، باید میزان و الگوی این مشکلات بررسی شود.

**هدف:**

شناسایی تعداد و درصد مقادیر گمشده در هر ستون، بررسی میزان تکرار کامل ردیف‌ها و مشخص‌کردن ستون‌هایی که بیشترین مشکل کیفیت داده را دارند.

**منطق تصمیم‌گیری:**

- مقادیر گمشده را به تفکیک ستون بررسی می‌کنیم.
- فقط درصد گمشدگی را ملاک تصمیم‌گیری قرار نمی‌دهیم؛ اهمیت هر ستون در تحلیل نیز مهم است.
- رکوردهای تکراری کامل را شناسایی می‌کنیم، اما هنوز حذفشان نمی‌کنیم.
- وضعیت `CustomerID` را جداگانه بررسی می‌کنیم، زیرا این ستون برای تحلیل رفتار مشتری و ساخت هر دو دیتاست ضروری است.

**فایده:**

نتایج این بررسی مبنای تصمیم‌گیری درباره نحوه برخورد با داده‌های ناقص و تکراری در مراحل بعدی Data Preparation خواهد بود.

</div>


In [6]:

import pandas as pd

# 1. تعداد و درصد مقادیر گمشده در هر ستون
missing_summary = pd.DataFrame({
    "Missing_Count": df.isna().sum(),
    "Missing_Percentage": (df.isna().mean() * 100).round(2)
})

missing_summary = missing_summary.sort_values(
    by="Missing_Percentage",
    ascending=False
)

print("Missing Values Summary:")
display(missing_summary)


# 2. بررسی تعداد و درصد رکوردهای تکراری کامل
duplicate_count = df.duplicated().sum()
duplicate_percentage = (duplicate_count / len(df)) * 100

print("\nDuplicate Records:")
print(f"Duplicate rows: {duplicate_count:,}")
print(f"Duplicate percentage: {duplicate_percentage:.2f}%")


# 3. بررسی وضعیت CustomerID
missing_customer_ids = df["CustomerID"].isna().sum()
known_customer_ids = df["CustomerID"].nunique()

print("\nCustomerID Analysis:")
print(f"Missing CustomerID rows: {missing_customer_ids:,}")
print(f"Known unique customers: {known_customer_ids:,}")
print(
    f"Missing CustomerID percentage: "
    f"{df['CustomerID'].isna().mean() * 100:.2f}%"
)


Missing Values Summary:


,Missing_Count,Missing_Percentage
CustomerID,135080,24.93
Description,1454,0.27
StockCode,0,0.00
InvoiceNo,0,0.00
Quantity,0,0.00
InvoiceDate,0,0.00
UnitPrice,0,0.00
Country,0,0.00



Duplicate Records:
Duplicate rows: 5,268
Duplicate percentage: 0.97%

CustomerID Analysis:
Missing CustomerID rows: 135,080
Known unique customers: 4,372
Missing CustomerID percentage: 24.93%



<div dir="rtl" align="right">

**تحلیل نتایج:**

بیشترین میزان مقادیر گمشده مربوط به `CustomerID` است. تعداد 135,080 ردیف فاقد شناسه مشتری هستند که معادل 24.93 درصد کل ردیف‌های دیتاست است.

این مسئله برای پروژه اهمیت زیادی دارد؛ زیرا هر دو مسیر اصلی مدل‌سازی در سطح مشتری انجام می‌شوند:

- در K-Means برای محاسبه ویژگی‌های رفتاری و LRFM باید تراکنش‌ها به مشتری مشخصی منتسب باشند.
- در XGBoost برای ساخت نمونه‌های آموزشی و تعیین وضعیت Churn باید سابقه خرید هر مشتری قابل شناسایی باشد.

بنابراین ردیف‌های فاقد `CustomerID` نمی‌توانند وارد تحلیل مشتری‌محور شوند؛ اما برای تحلیل عمومی تراکنش‌ها همچنان قابل استفاده‌اند.

ستون `Description` نیز دارای 1,454 مقدار گمشده، معادل 0.27 درصد است. از آنجا که `StockCode` در تمام ردیف‌ها موجود است، نبود توضیحات کالا به‌تنهایی دلیل کافی برای حذف تراکنش نیست.

**۲. نتایج بررسی رکوردهای تکراری**

- تعداد ردیف‌های کاملاً تکراری اضافه: 5,268
- درصد ردیف‌های تکراری اضافه: 0.97 درصد
- تعداد ردیف‌های دارای CustomerID گمشده: 135,080
- تعداد مشتریان یکتای شناسایی‌شده: 4,372

**تحلیل نتایج:**

بررسی اولیه نشان داد که 5,268 ردیف کاملاً تکراری در دیتاست وجود دارد. این ردیف‌ها ممکن است باعث محاسبه چندباره مقدار خرید و در نتیجه انحراف ویژگی‌های مشتری شوند.

همچنین باید توجه داشت که تعداد 135,080 مربوط به ردیف‌های فاقد شناسه مشتری است، نه تعداد مشتریان گمشده یا ناشناس.

**۳. تصمیم‌گیری درباره مقادیر گمشده**

- `CustomerID`: عدم جایگزینی با شناسه ساختگی؛ نگهداری در داده خام و کنارگذاری از تحلیل مشتری‌محور.
- `Description`: نگهداری ردیف‌های دارای مقدار گمشده؛ عدم حذف تراکنش صرفاً به دلیل نبود توضیحات کالا.
- سایر ستون‌ها: در این بررسی مقدار گمشده مشاهده نشد.

**۴. تصمیم‌گیری درباره رکوردهای تکراری**

رکوردهای کاملاً تکراری باید از نسخه تحلیلی حذف شوند، اما دیتاست خام بدون تغییر حفظ می‌شود.

این حذف فقط برای ردیف‌هایی انجام می‌شود که تمام ستون‌های آن‌ها با یکدیگر یکسان باشند.

**نتیجه:**

دو مشکل اصلی کیفیت داده شناسایی شد:

1. نبود شناسه مشتری در 24.93 درصد از ردیف‌های تراکنش.
2. وجود 5,268 ردیف تکراری اضافه.

تصمیم اولیه برای مدیریت این مشکلات مشخص شد؛ اما اجرای حذف رکوردهای تکراری و ساخت نسخه پاک‌سازی‌شده در مرحله اجرایی بعدی انجام می‌شود.

</div>


---


<div dir="rtl" align="right">

### سؤال سوم: آیا رکوردهای تکراری باید حذف شوند؟

**چرایی و دلیل:**

رکوردهای تکراری کامل ممکن است باعث محاسبه چندباره یک تراکنش شوند. این مسئله می‌تواند روی مجموع مقدار خرید، ارزش خرید مشتری، تعداد اقلام و در نهایت ویژگی‌های مورد استفاده در مدل‌های Clustering و Classification اثر بگذارد.

بااین‌حال، پیش از حذف باید ماهیت رکوردهای تکراری را بررسی کنیم. ممکن است یک کالا در یک فاکتور چند بار با اطلاعات یکسان ثبت شده باشد؛ بنابراین صرفاً مشاهده مقادیر یکسان، بدون بررسی زمینه داده، برای نتیجه‌گیری قطعی کافی نیست.

**هدف:**

شناسایی ردیف‌های تکراری، مشاهده نمونه‌های آن‌ها، بررسی محل وقوع تکرار و اندازه‌گیری اثر احتمالی حذف آن‌ها بر مقدار خام تراکنش‌ها.

**منطق تصمیم‌گیری:**

- ردیف‌های تکراری را با مقایسه تمام ستون‌ها شناسایی می‌کنیم.
- نمونه ردیف‌های تکراری را مشاهده می‌کنیم.
- تعداد فاکتورهای درگیر در تکرار را بررسی می‌کنیم.
- اثر ردیف‌های تکراری اضافه را بر مجموع Quantity و حاصل‌ضرب Quantity در UnitPrice محاسبه می‌کنیم.
- پس از بررسی، درباره حذف ردیف‌های کاملاً تکراری تصمیم می‌گیریم.

**فایده:**

جلوگیری از محاسبه چندباره تراکنش‌ها و کاهش احتمال انحراف ویژگی‌های مشتری در مراحل بعدی Data Preparation.

</div>


In [8]:

# 1. شناسایی تمام ردیف‌هایی که در گروه‌های تکراری قرار دارند
duplicate_groups = df[df.duplicated(keep=False)].copy()

# 2. شناسایی فقط نسخه‌های اضافه رکوردهای تکراری
# این همان 5268 ردیفی است که در بررسی قبلی شمارش کردیم.
extra_duplicates = df[df.duplicated(keep="first")].copy()

print("Duplicate Analysis:")
print(f"Rows involved in duplicate groups: {len(duplicate_groups):,}")
print(f"Extra duplicate rows: {len(extra_duplicates):,}")
print(f"Number of affected invoices: {extra_duplicates['InvoiceNo'].nunique():,}")


# 3. مشاهده نمونه‌ای از رکوردهای تکراری
print("\nDuplicate Samples:")
display(
    duplicate_groups.sort_values(
        by=["InvoiceNo", "StockCode"]
    ).head(20)
)


# 4. محاسبه اثر رکوردهای تکراری اضافه بر داده خام
extra_duplicates["RawLineValue"] = (
    extra_duplicates["Quantity"] * extra_duplicates["UnitPrice"]
)

print("\nPotential Impact of Extra Duplicate Rows:")
print(f"Extra duplicate Quantity: {extra_duplicates['Quantity'].sum():,}")
print(
    "Extra duplicate raw line value: "
    f"{extra_duplicates['RawLineValue'].sum():,.2f}"
)
print(
    "Absolute raw line value: "
    f"{extra_duplicates['RawLineValue'].abs().sum():,.2f}"
)


# 5. بررسی فاکتورهایی که بیشترین تعداد ردیف تکراری اضافه را دارند
print("\nInvoices with the Most Extra Duplicate Rows:")

display(
    extra_duplicates.groupby("InvoiceNo")
    .size()
    .sort_values(ascending=False)
    .head(10)
    .rename("Extra_Duplicate_Rows")
    .to_frame()
)


Duplicate Analysis:
Rows involved in duplicate groups: 10,147
Extra duplicate rows: 5,268
Number of affected invoices: 1,933

Duplicate Samples:


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
494,536409,21866,UNION JACK FLAG LUGGAGE TAG,1,12/1/10 11:45,1.25,17908.0,United Kingdom
517,536409,21866,UNION JACK FLAG LUGGAGE TAG,1,12/1/10 11:45,1.25,17908.0,United Kingdom
485,536409,22111,SCOTTIE DOG HOT WATER BOTTLE,1,12/1/10 11:45,4.95,17908.0,United Kingdom
539,536409,22111,SCOTTIE DOG HOT WATER BOTTLE,1,12/1/10 11:45,4.95,17908.0,United Kingdom
489,536409,22866,HAND WARMER SCOTTY DOG DESIGN,1,12/1/10 11:45,2.10,17908.0,United Kingdom
527,536409,22866,HAND WARMER SCOTTY DOG DESIGN,1,12/1/10 11:45,2.10,17908.0,United Kingdom
521,536409,22900,SET 2 TEA TOWELS I LOVE LONDON,1,12/1/10 11:45,2.95,17908.0,United Kingdom
537,536409,22900,SET 2 TEA TOWELS I LOVE LONDON,1,12/1/10 11:45,2.95,17908.0,United Kingdom
565,536412,21448,12 DAISY PEGS IN WOOD BOX,2,12/1/10 11:49,1.65,17920.0,United Kingdom
578,536412,21448,12 DAISY PEGS IN WOOD BOX,1,12/1/10 11:49,1.65,17920.0,United Kingdom



Potential Impact of Extra Duplicate Rows:
Extra duplicate Quantity: 13,948
Extra duplicate raw line value: 21,740.98
Absolute raw line value: 27,406.50

Invoices with the Most Extra Duplicate Rows:


,Extra_Duplicate_Rows
InvoiceNo,
555524,34
547651,31
579458,28
574481,27
572103,27
577482,26
537781,25
552285,24
575117,21



<div dir="rtl" align="right">

### نتایج اجرا و تحلیل: بررسی ماهیت و اثر رکوردهای تکراری

در بررسی دقیق‌تر رکوردهای تکراری، دو شاخص متفاوت محاسبه شد:

- تعداد ردیف‌های عضو گروه‌های تکراری: 10,147
- تعداد نسخه‌های تکراری اضافه: 5,268

تفاوت این دو مقدار به دلیل وجود گروه‌هایی است که یک رکورد در آن‌ها بیش از دو بار تکرار شده است. در محاسبه نسخه‌های اضافه، اولین نمونه از هر گروه حفظ می‌شود.

**۱. گستردگی تکرارها**

تعداد فاکتورهای درگیر در تکرار: 1,933

بنابراین رکوردهای تکراری فقط به چند فاکتور محدود نیستند و در تعداد قابل‌توجهی از فاکتورها مشاهده شده‌اند.

**۳. تصمیم نهایی**

نسخه‌های کاملاً تکراری از داده تحلیلی حذف می‌شوند و اولین نمونه از هر گروه باقی می‌ماند.

ردیف‌هایی که فقط در بعضی ویژگی‌ها مانند `InvoiceNo` یا `StockCode` مشترک هستند، تکراری کامل محسوب نمی‌شوند و نباید با این روش حذف شوند.

**نتیجه:**

با توجه به وجود 5,268 نسخه تکراری اضافه و اثر بالقوه آن‌ها بر محاسبات، حذف تکرارهای کامل از نسخه تحلیلی تصمیم مناسبی است.

</div>


In [9]:

# ساخت نسخه جدید بدون رکوردهای کاملاً تکراری
df_dedup = df.drop_duplicates(keep="first").copy()

# بررسی نتیجه
print("Original Dataset Shape:", df.shape)
print("Dataset Shape After Deduplication:", df_dedup.shape)

print(
    "Removed Rows:",
    len(df) - len(df_dedup)
)

print(
    "Remaining Duplicate Rows:",
    df_dedup.duplicated().sum()
)


Original Dataset Shape: (541909, 8)
Dataset Shape After Deduplication: (536641, 8)
Removed Rows: 5268
Remaining Duplicate Rows: 0



<div dir="rtl" align="right">

### نتایج اجرا و تحلیل سؤال سوم: حذف رکوردهای کاملاً تکراری

پس از شناسایی و بررسی رکوردهای تکراری، تصمیم گرفته شد نسخه‌های کاملاً تکراری از داده تحلیلی حذف شوند و اولین نمونه از هر گروه حفظ شود.

برای اجرای این تصمیم، از متد `drop_duplicates()` با مقدار `keep="first"` استفاده شد. همچنین برای جلوگیری از تغییر دیتاست خام، یک کپی مستقل از داده‌ها ایجاد شد.

**تحلیل نتایج:**

تعداد ردیف‌های دیتاست از 541,909 به 536,641 کاهش یافت. این کاهش دقیقاً برابر با 5,268 ردیف تکراری اضافه شناسایی‌شده در مرحله بررسی بود.

پس از اجرای عملیات، مقدار `duplicated().sum()` برابر صفر شد. بنابراین هیچ ردیف کاملاً تکراری بر اساس تمام ستون‌ها در نسخه جدید باقی نمانده است.

تعداد ستون‌ها بدون تغییر باقی ماند؛ زیرا عملیات حذف تکرار فقط روی ردیف‌های دیتاست انجام شده است.

**تصمیم نهایی:**

نسخه تحلیلی بدون رکوردهای کاملاً تکراری با نام `df_dedup` ساخته شد. دیتاست خام با نام `df` نیز برای حفظ قابلیت بازبینی و مقایسه نگهداری شد.

**فایده برای پروژه:**

این عملیات از محاسبه چندباره ردیف‌های یکسان جلوگیری می‌کند و احتمال انحراف ویژگی‌هایی مانند مجموع خرید، تعداد کالاهای خریداری‌شده و ارزش تراکنش مشتری را کاهش می‌دهد.

این نسخه، مبنای ادامه مراحل پاک‌سازی داده خواهد بود.

</div>


---


<div dir="rtl" align="right">

### سؤال چهارم: فاکتورهای لغوشده و تراکنش‌های دارای مقادیر منفی یا صفر چه وضعیتی دارند؟

**چرایی و دلیل:**

در دیتاست Online Retail تمام ردیف‌ها الزاماً نشان‌دهنده خرید موفق مشتری نیستند. برخی فاکتورها لغو شده‌اند، برخی ردیف‌ها دارای تعداد کالای منفی هستند و در بعضی موارد قیمت واحد نیز منفی یا صفر است.

این مقادیر می‌توانند نشان‌دهنده رویدادهای متفاوتی مانند لغو سفارش، مرجوعی کالا، اصلاح اطلاعات یا ثبت‌های مالی خاص باشند. بنابراین حذف تمام ردیف‌های دارای مقادیر منفی بدون بررسی، ممکن است باعث از بین رفتن اطلاعات ارزشمند شود.

این موضوع برای پروژه اهمیت ویژه‌ای دارد؛ زیرا ویژگی‌های رفتاری مشتری مانند Monetary، Frequency، Recency و در نهایت برچسب Churn باید بر اساس تعریف مشخصی از خرید معتبر ساخته شوند.

**هدف:**

شناسایی و اندازه‌گیری موارد زیر:

1. فاکتورهای لغوشده بر اساس پیشوند `C` در `InvoiceNo`.
2. ردیف‌های دارای `Quantity` منفی یا صفر.
3. ردیف‌های دارای `UnitPrice` منفی یا صفر.
4. میزان هم‌پوشانی این شرایط با یکدیگر.
5. نمونه‌هایی از هر گروه برای بررسی ماهیت داده‌ها.

**منطق تصمیم‌گیری:**

- فاکتورهایی که شماره آن‌ها با `C` شروع می‌شود، به‌عنوان فاکتور لغوشده شناسایی می‌شوند.
- تعداد منفی می‌تواند نشان‌دهنده مرجوعی یا ثبت اصلاحی باشد؛ اما به‌تنهایی برای اثبات مرجوعی کافی نیست.
- قیمت منفی ممکن است مربوط به تعدیلات یا ثبت‌های خاص باشد و باید جداگانه بررسی شود.
- مقدار صفر در تعداد یا قیمت نیز باید از مقادیر منفی تفکیک شود.
- هم‌پوشانی شرایط بررسی می‌شود تا از شمارش چندباره یک ردیف جلوگیری شود.

**فایده:**

این بررسی به ما کمک می‌کند خریدهای معتبر را از لغو سفارش‌ها و سایر ثبت‌های غیرمعمول جدا کنیم و در مراحل بعدی، ویژگی‌های مشتری و برچسب Churn را بر اساس تعریف قابل دفاعی بسازیم.

</div>


In [10]:

# -----------------------------------------
# 1. ساخت ماسک‌های شناسایی وضعیت تراکنش‌ها
# -----------------------------------------

# فاکتورهایی که شماره آن‌ها با C شروع می‌شود
cancel_mask = (
    df_dedup["InvoiceNo"]
    .str.startswith("C", na=False)
)

# تعداد منفی و صفر
negative_quantity_mask = df_dedup["Quantity"] < 0
zero_quantity_mask = df_dedup["Quantity"] == 0

# قیمت واحد منفی و صفر
negative_price_mask = df_dedup["UnitPrice"] < 0
zero_price_mask = df_dedup["UnitPrice"] == 0


# -----------------------------------------
# 2. ساخت جدول خلاصه وضعیت‌ها
# -----------------------------------------

conditions = {
    "Cancelled Invoices": cancel_mask,
    "Negative Quantity": negative_quantity_mask,
    "Zero Quantity": zero_quantity_mask,
    "Negative UnitPrice": negative_price_mask,
    "Zero UnitPrice": zero_price_mask,
    "Non-positive Quantity": df_dedup["Quantity"] <= 0,
    "Non-positive UnitPrice": df_dedup["UnitPrice"] <= 0,
}

summary = []

for condition_name, mask in conditions.items():
    subset = df_dedup.loc[mask]

    summary.append({
        "Condition": condition_name,
        "Rows": len(subset),
        "Percentage": round(mask.mean() * 100, 2),
        "Unique Invoices": subset["InvoiceNo"].nunique(),
        "Unique Customers": subset["CustomerID"].nunique()
    })

condition_summary = pd.DataFrame(summary)

print("Transaction Quality Summary:")
display(condition_summary)


# -----------------------------------------
# 3. بررسی هم‌پوشانی شرایط
# -----------------------------------------

overlap_summary = pd.DataFrame({
    "Overlap": [
        "Cancelled Invoice & Negative Quantity",
        "Cancelled Invoice & Negative UnitPrice",
        "Negative Quantity & Negative UnitPrice",
        "Cancelled Invoice & Non-positive Quantity",
        "Cancelled Invoice & Non-positive UnitPrice",
    ],
    "Rows": [
        (cancel_mask & negative_quantity_mask).sum(),
        (cancel_mask & negative_price_mask).sum(),
        (negative_quantity_mask & negative_price_mask).sum(),
        (cancel_mask & (df_dedup["Quantity"] <= 0)).sum(),
        (cancel_mask & (df_dedup["UnitPrice"] <= 0)).sum(),
    ]
})

print("\nOverlap Analysis:")
display(overlap_summary)


# -----------------------------------------
# 4. مشاهده نمونه ردیف‌های هر گروه
# -----------------------------------------

sample_columns = [
    "InvoiceNo",
    "StockCode",
    "Description",
    "Quantity",
    "InvoiceDate",
    "UnitPrice",
    "CustomerID",
    "Country"
]

print("\nCancelled Invoice Samples:")
display(
    df_dedup.loc[cancel_mask, sample_columns].head(10)
)

print("\nNegative Quantity Samples:")
display(
    df_dedup.loc[negative_quantity_mask, sample_columns].head(10)
)

print("\nNegative UnitPrice Samples:")
display(
    df_dedup.loc[negative_price_mask, sample_columns].head(10)
)

print("\nZero Quantity Samples:")
display(
    df_dedup.loc[zero_quantity_mask, sample_columns].head(10)
)

print("\nZero UnitPrice Samples:")
display(
    df_dedup.loc[zero_price_mask, sample_columns].head(10)
)


Transaction Quality Summary:


,Condition,Rows,Percentage,Unique Invoices,Unique Customers
0,Cancelled Invoices,9251,1.72,3836,1589
1,Negative Quantity,10587,1.97,5172,1589
2,Zero Quantity,0,0.00,0,0
3,Negative UnitPrice,2,0.00,2,0
4,Zero UnitPrice,2510,0.47,2155,31
5,Non-positive Quantity,10587,1.97,5172,1589
6,Non-positive UnitPrice,2512,0.47,2157,31



Overlap Analysis:


,Overlap,Rows
0,Cancelled Invoice & Negative Quantity,9251
1,Cancelled Invoice & Negative UnitPrice,0
2,Negative Quantity & Negative UnitPrice,0
3,Cancelled Invoice & Non-positive Quantity,9251
4,Cancelled Invoice & Non-positive UnitPrice,0



Cancelled Invoice Samples:


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
141,C536379,D,Discount,-1,12/1/10 9:41,27.50,14527.0,United Kingdom
154,C536383,35004C,SET OF 3 COLOURED FLYING DUCKS,-1,12/1/10 9:49,4.65,15311.0,United Kingdom
235,C536391,22556,PLASTERS IN TIN CIRCUS PARADE,-12,12/1/10 10:24,1.65,17548.0,United Kingdom
236,C536391,21984,PACK OF 12 PINK PAISLEY TISSUES,-24,12/1/10 10:24,0.29,17548.0,United Kingdom
237,C536391,21983,PACK OF 12 BLUE PAISLEY TISSUES,-24,12/1/10 10:24,0.29,17548.0,United Kingdom
238,C536391,21980,PACK OF 12 RED RETROSPOT TISSUES,-24,12/1/10 10:24,0.29,17548.0,United Kingdom
239,C536391,21484,CHICK GREY HOT WATER BOTTLE,-12,12/1/10 10:24,3.45,17548.0,United Kingdom
240,C536391,22557,PLASTERS IN TIN VINTAGE PAISLEY,-12,12/1/10 10:24,1.65,17548.0,United Kingdom
241,C536391,22553,PLASTERS IN TIN SKULLS,-24,12/1/10 10:24,1.65,17548.0,United Kingdom
939,C536506,22960,JAM MAKING SET WITH JARS,-6,12/1/10 12:38,4.25,17897.0,United Kingdom


... (4 outputs left out)


Zero Quantity Samples:


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country



Zero UnitPrice Samples:


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
622,536414,22139,NaN,56,12/1/10 11:52,0.0,NaN,United Kingdom
1970,536545,21134,NaN,1,12/1/10 14:32,0.0,NaN,United Kingdom
1971,536546,22145,NaN,1,12/1/10 14:33,0.0,NaN,United Kingdom
1972,536547,37509,NaN,1,12/1/10 14:33,0.0,NaN,United Kingdom
1987,536549,85226A,NaN,1,12/1/10 14:34,0.0,NaN,United Kingdom
1988,536550,85044,NaN,1,12/1/10 14:34,0.0,NaN,United Kingdom
2024,536552,20950,NaN,1,12/1/10 14:34,0.0,NaN,United Kingdom
2025,536553,37461,NaN,3,12/1/10 14:35,0.0,NaN,United Kingdom
2026,536554,84670,NaN,23,12/1/10 14:35,0.0,NaN,United Kingdom
2406,536589,21777,NaN,-10,12/1/10 16:50,0.0,NaN,United Kingdom



<div dir="rtl" align="right">

### نتایج اجرا و تحلیل سؤال چهارم: بررسی فاکتورهای لغوشده و مقادیر منفی یا صفر

پس از اجرای بررسی روی دیتاست `df_dedup`، وضعیت فاکتورهای لغوشده، تعداد کالا و قیمت واحد مشخص شد.

**۲. تحلیل فاکتورهای لغوشده**

تعداد 9,251 ردیف متعلق به فاکتورهایی است که شماره آن‌ها با حرف `C` شروع می‌شود. این ردیف‌ها به 3,836 شماره فاکتور یکتا تعلق دارند.

بنابراین باید میان تعداد خطوط تراکنش لغوشده و تعداد فاکتورهای لغوشده تفاوت قائل شد.

**۳. تحلیل Quantity منفی**

تعداد 10,587 ردیف دارای مقدار منفی در ستون `Quantity` بودند.

بررسی هم‌پوشانی نشان داد که تمام 9,251 ردیف فاکتورهای لغوشده در این گروه قرار دارند.

در نتیجه، تعداد 1,336 ردیف دارای Quantity منفی خارج از فاکتورهای لغوشده قرار می‌گیرند.

این ردیف‌ها می‌توانند نشان‌دهنده مرجوعی کالا یا سایر اصلاحات تراکنشی باشند؛ بنابراین صرفاً بر اساس منفی بودن Quantity نمی‌توان ماهیت دقیق آن‌ها را تعیین کرد.

**۴. تحلیل UnitPrice منفی**

تنها دو ردیف دارای قیمت واحد منفی شناسایی شدند. در هر دو نمونه، مقدار Description برابر `Adjust bad debt` و مقدار StockCode برابر `B` بود.

همچنین CustomerID هر دو ردیف گمشده بود.

این الگو نشان می‌دهد که این دو ردیف به احتمال زیاد ثبت‌های تعدیلی حسابداری هستند و نباید به‌عنوان خرید یا مرجوعی معمول کالا در محاسبات مشتری‌محور استفاده شوند.

**۵. تحلیل UnitPrice صفر**

تعداد 2,510 ردیف دارای قیمت واحد صفر بودند که در 2,155 فاکتور مشاهده شدند.

از آنجا که این ردیف‌ها ممکن است مربوط به کالاهای رایگان، هدیه یا سایر ثبت‌های خاص باشند، نمی‌توان تمام آن‌ها را بدون بررسی بیشتر نامعتبر تلقی کرد.

**۶. تحلیل هم‌پوشانی**

بررسی هم‌پوشانی نشان داد:

- تعداد 9,251 ردیف به‌طور هم‌زمان دارای فاکتور لغوشده و Quantity منفی بودند.
- هیچ ردیفی به‌طور هم‌زمان دارای فاکتور لغوشده و UnitPrice منفی نبود.
- هیچ ردیفی به‌طور هم‌زمان دارای Quantity منفی و UnitPrice منفی نبود.

**تصمیم نهایی:**

1. فاکتورهای لغوشده از مجموعه خریدهای معتبر مشتریان کنار گذاشته خواهند شد.
2. ردیف‌های دارای Quantity منفی که فاکتور لغوشده ندارند، جداگانه برای شناسایی مرجوعی‌ها بررسی می‌شوند.
3. دو ردیف دارای UnitPrice منفی به‌عنوان ثبت‌های تعدیلی نگهداری می‌شوند، اما وارد محاسبات خرید مشتری نخواهند شد.
4. ردیف‌های دارای UnitPrice صفر پیش از هرگونه حذف یا کنارگذاری بیشتر بررسی می‌شوند.
5. هیچ‌یک از این دسته‌ها در این مرحله از دیتاست خام حذف نمی‌شوند.

**فایده برای پروژه:**

این تفکیک از وارد شدن فاکتورهای لغوشده، تعدیلات مالی و تراکنش‌های غیرمعمول به محاسبات خرید جلوگیری می‌کند و امکان بررسی جداگانه رفتار مرجوعی مشتریان را فراهم می‌سازد.

</div>


In [11]:

# شناسایی ردیف‌های دارای Quantity منفی
# که فاکتورشان با C شروع نمی‌شود

non_cancelled_negative_qty = df_dedup.loc[
    (df_dedup["Quantity"] < 0)
    & (~df_dedup["InvoiceNo"].str.startswith("C", na=False))
].copy()

print("Non-cancelled Negative Quantity Analysis:")

print("Rows:", len(non_cancelled_negative_qty))
print(
    "Unique Invoices:",
    non_cancelled_negative_qty["InvoiceNo"].nunique()
)
print(
    "Unique Customers:",
    non_cancelled_negative_qty["CustomerID"].nunique()
)
print(
    "Missing CustomerID:",
    non_cancelled_negative_qty["CustomerID"].isna().sum()
)

# مشاهده نمونه‌ها
display(
    non_cancelled_negative_qty[
        [
            "InvoiceNo",
            "StockCode",
            "Description",
            "Quantity",
            "InvoiceDate",
            "UnitPrice",
            "CustomerID",
            "Country"
        ]
    ].head(20)
)


# بررسی وضعیت قیمت در همین ردیف‌ها
print("\nUnitPrice Distribution:")

display(
    non_cancelled_negative_qty["UnitPrice"]
    .describe()
    .to_frame()
)

# بررسی تعداد خطوط بر اساس کد کالا
print("\nMost Frequent StockCodes:")

display(
    non_cancelled_negative_qty["StockCode"]
    .value_counts()
    .head(15)
    .to_frame("Rows")
)


Non-cancelled Negative Quantity Analysis:
Rows: 1336
Unique Invoices: 1336
Unique Customers: 0
Missing CustomerID: 1336


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
2406,536589,21777,NaN,-10,12/1/10 16:50,0.0,NaN,United Kingdom
4347,536764,84952C,NaN,-38,12/2/10 14:42,0.0,NaN,United Kingdom
7188,536996,22712,NaN,-20,12/3/10 15:30,0.0,NaN,United Kingdom
7189,536997,22028,NaN,-20,12/3/10 15:30,0.0,NaN,United Kingdom
7190,536998,85067,NaN,-6,12/3/10 15:30,0.0,NaN,United Kingdom
7192,537000,21414,NaN,-22,12/3/10 15:32,0.0,NaN,United Kingdom
7193,537001,21653,NaN,-6,12/3/10 15:33,0.0,NaN,United Kingdom
7195,537003,85126,NaN,-2,12/3/10 15:33,0.0,NaN,United Kingdom
7196,537004,21814,NaN,-30,12/3/10 15:34,0.0,NaN,United Kingdom
7197,537005,21692,NaN,-70,12/3/10 15:35,0.0,NaN,United Kingdom



UnitPrice Distribution:


,UnitPrice
count,1336.0
mean,0.0
std,0.0
min,0.0
25%,0.0
50%,0.0
75%,0.0
max,0.0



Most Frequent StockCodes:


,Rows
StockCode,
85175,5
21830,5
85172,4
22719,4
82494L,4
72802C,4
21161,3
22162,3
22034,3



<div dir="rtl" align="right">

### نتایج اجرا و تحلیل سؤال چهارم: بررسی فاکتورهای لغوشده و تراکنش‌های دارای مقادیر منفی یا صفر

در این مرحله وضعیت فاکتورهای لغوشده و مقادیر غیرمعمول در ستون‌های `Quantity` و `UnitPrice` بررسی شد.

داده‌های مورد بررسی مربوط به دیتاست پس از حذف رکوردهای کاملاً تکراری، شامل 536,641 ردیف بود.

#### ۱. نتایج بررسی وضعیت تراکنش‌ها

| وضعیت | تعداد ردیف | درصد کل | فاکتور یکتا | مشتری یکتا |
|---|---:|---:|---:|---:|
| Cancelled Invoices | 9,251 | 1.72% | 3,836 | 1,589 |
| Negative Quantity | 10,587 | 1.97% | 5,172 | 1,589 |
| Zero Quantity | 0 | 0% | 0 | 0 |
| Negative UnitPrice | 2 | 0.00% | 2 | 0 |
| Zero UnitPrice | 2,510 | 0.47% | 2,155 | 31 |

#### ۲. بررسی فاکتورهای لغوشده

تعداد 9,251 ردیف دارای شماره فاکتور با پیشوند `C` شناسایی شد. این ردیف‌ها متعلق به 3,836 فاکتور یکتا هستند.

بنابراین تعداد ردیف‌های لغوشده با تعداد فاکتورهای لغوشده یکسان نیست؛ زیرا هر فاکتور می‌تواند چندین سطر کالایی داشته باشد.

#### ۳. بررسی Quantity منفی

تعداد 10,587 ردیف دارای مقدار منفی در ستون `Quantity` بودند.

بررسی هم‌پوشانی نشان داد که تمام 9,251 ردیف فاکتورهای لغوشده دارای Quantity منفی هستند.

در نتیجه 1,336 ردیف دارای Quantity منفی خارج از فاکتورهای لغوشده قرار گرفتند.

#### ۴. بررسی تکمیلی 1,336 ردیف دارای Quantity منفی

این ردیف‌ها به‌صورت مستقل بررسی شدند و نتایج زیر به دست آمد:

| شاخص | مقدار |
|---|---:|
| تعداد ردیف‌ها | 1,336 |
| تعداد فاکتورهای یکتا | 1,336 |
| تعداد مشتریان یکتا | 0 |
| تعداد CustomerID گمشده | 1,336 |
| میانگین UnitPrice | 0 |
| انحراف معیار UnitPrice | 0 |
| حداقل UnitPrice | 0 |
| حداکثر UnitPrice | 0 |

بررسی نشان داد که تمام 1,336 ردیف دارای قیمت واحد صفر و شناسه مشتری گمشده هستند.

همچنین تعداد فاکتورهای یکتا با تعداد ردیف‌ها برابر است؛ بنابراین هر ردیف به یک شماره فاکتور مستقل تعلق دارد.

با توجه به نبود شناسه مشتری و صفر بودن قیمت، امکان انتساب این تراکنش‌ها به سابقه خرید یا رفتار مرجوعی مشتری مشخص وجود ندارد.

از طرف دیگر، صرفاً بر اساس منفی بودن Quantity نمی‌توان ماهیت این ردیف‌ها را مرجوعی قطعی در نظر گرفت.

بنابراین این گروه با عنوان `Unidentified Negative-Quantity Records` شناسایی می‌شود.

#### ۵. بررسی UnitPrice منفی

دو ردیف دارای قیمت واحد منفی شناسایی شدند.

در هر دو ردیف:

- `StockCode` برابر `B` بود.
- `Description` برابر `Adjust bad debt` بود.
- `Quantity` برابر 1 بود.
- `UnitPrice` برابر 11062.06- بود.
- `CustomerID` گمشده بود.

این مشخصات نشان می‌دهد که این ردیف‌ها با ثبت‌های تعدیلی بدهی ارتباط دارند و نباید به‌عنوان خرید یا مرجوعی معمول کالا وارد محاسبات مشتری‌محور شوند.

#### ۶. بررسی UnitPrice صفر

تعداد 2,510 ردیف دارای UnitPrice صفر شناسایی شد که در 2,155 فاکتور قرار داشتند.

از آنجا که این ردیف‌ها ممکن است مربوط به کالاهای رایگان، هدیه یا ثبت‌های خاص باشند، صفر بودن قیمت به‌تنهایی دلیل کافی برای حذف تمام آن‌ها نیست.

این دسته به بررسی جداگانه نیاز دارد؛ به‌ویژه در مواردی که CustomerID موجود است.

نتایج نشان می‌دهد که تمام ردیف‌های فاکتورهای لغوشده دارای Quantity منفی هستند؛ اما تمام ردیف‌های دارای Quantity منفی، فاکتور لغوشده نیستند.

#### ۸. تصمیم نهایی

1. فاکتورهای لغوشده از مجموعه خریدهای معتبر مشتریان کنار گذاشته می‌شوند.
2. ردیف‌های دارای Quantity منفی خارج از فاکتورهای لغوشده به‌صورت یک گروه مستقل نگهداری می‌شوند و تا زمان شناسایی ماهیتشان، مرجوعی قطعی محسوب نمی‌شوند.
3. دو ردیف دارای UnitPrice منفی به‌عنوان ثبت‌های تعدیلی شناسایی می‌شوند و وارد محاسبات خرید مشتری نخواهند شد.
4. ردیف‌های دارای UnitPrice صفر به‌صورت خودکار حذف نمی‌شوند و برای بررسی بیشتر نگهداری می‌شوند.
5. اطلاعات این گروه‌ها در داده خام حفظ می‌شود و حذف یا کنارگذاری آن‌ها فقط در زمان ساخت دیتاست متناسب با هر مسئله انجام خواهد شد.

**فایده برای پروژه:**

این تفکیک از ورود فاکتورهای لغوشده، ثبت‌های تعدیلی و تراکنش‌های فاقد هویت مشتری به محاسبات خرید جلوگیری می‌کند. همچنین امکان نگهداری مستقل اطلاعات مربوط به مرجوعی و ثبت‌های غیرمعمول را فراهم می‌سازد.

</div>


In [12]:

# بررسی کامل 1336 ردیف با Quantity منفی خارج از فاکتور لغوشده

audit = non_cancelled_negative_qty

description_check = pd.DataFrame({
    "Missing_Count": audit.isna().sum(),
    "Missing_Percentage": (audit.isna().mean() * 100).round(2)
})

print("Missing Values in Unidentified Records:")
display(description_check)

print("\nStockCode and Description Samples:")

display(
    audit.groupby(
        ["StockCode", "Description"],
        dropna=False
    )
    .size()
    .sort_values(ascending=False)
    .head(20)
    .rename("Rows")
    .to_frame()
)


Missing Values in Unidentified Records:


,Missing_Count,Missing_Percentage
InvoiceNo,0,0.00
StockCode,0,0.00
Description,862,64.52
Quantity,0,0.00
InvoiceDate,0,0.00
UnitPrice,0,0.00
CustomerID,1336,100.00
Country,0,0.00



StockCode and Description Samples:


,,Rows
StockCode,Description,
21161,NaN,3
22162,NaN,3
22034,NaN,3
20892,NaN,3
20966,NaN,3
22114,NaN,3
72802B,check,3
72802C,check,3
84631,NaN,3



<div dir="rtl" align="right">

#### ۹. بررسی تکمیلی مقادیر گمشده در رکوردهای دارای Quantity منفی خارج از فاکتورهای لغوشده

به‌منظور بررسی دقیق‌تر 1,336 ردیف دارای Quantity منفی که شماره فاکتور آن‌ها با حرف C شروع نمی‌شد، وضعیت مقادیر گمشده و ترکیب کد کالا و توضیحات بررسی شد.

**نتایج بررسی Missing Values:**

| ستون | تعداد مقادیر گمشده | درصد گمشدگی |
|---|---:|---:|
| InvoiceNo | 0 | 0% |
| StockCode | 0 | 0% |
| Description | 862 | 64.52% |
| Quantity | 0 | 0% |
| InvoiceDate | 0 | 0% |
| UnitPrice | 0 | 0% |
| CustomerID | 1,336 | 100% |
| Country | 0 | 0% |

نتایج نشان داد که تمام 1,336 ردیف فاقد `CustomerID` هستند. همچنین در 862 ردیف مقدار `Description` گمشده است؛ درحالی‌که 474 ردیف دارای توضیحات کالا هستند.

بنابراین، نبود توضیحات کالا در تمام این گروه یکسان نیست و نمی‌توان تمام ردیف‌ها را از نظر اطلاعات توصیفی در یک وضعیت قرار داد.

**بررسی StockCode و Description:**

در بررسی ترکیب کد کالا و توضیحات، نمونه‌هایی مشاهده شد که مقدار `Description` برابر `check` بود؛ از جمله کدهای `72802B` و `72802C`.

این موارد برای بررسی کدهای خاص و ثبت‌های غیرمعمول در مراحل بعد نگهداری می‌شوند و بدون شواهد بیشتر به‌عنوان کالای معمولی یا غیرکالایی طبقه‌بندی نمی‌شوند.

**تحلیل نهایی:**

با توجه به اینکه تمام این ردیف‌ها فاقد شناسه مشتری هستند، امکان اتصال آن‌ها به سوابق خرید مشتریان وجود ندارد. بنابراین نمی‌توان از آن‌ها برای محاسبه LRFM، ساخت ویژگی‌های مشتری یا تعیین وضعیت Churn استفاده کرد.

از طرف دیگر، وجود `StockCode` و اطلاعات زمانی این امکان را فراهم می‌کند که این رکوردها در بررسی عمومی کیفیت داده و شناسایی کدهای خاص حفظ شوند.

**تصمیم نهایی:**

این 1,336 ردیف در داده خام نگهداری می‌شوند، اما در ساخت دیتاست‌های مشتری‌محور مورد استفاده قرار نمی‌گیرند. همچنین برای آن‌ها شناسه مشتری ساختگی یا توضیحات جایگزین تولید نمی‌شود.

**نتیجه سؤال چهارم:**

فاکتورهای لغوشده، تراکنش‌های دارای تعداد منفی، تعدیلات دارای قیمت منفی، ردیف‌های دارای قیمت صفر و رکوردهای منفی فاقد هویت مشتری از یکدیگر تفکیک شدند. این تفکیک مبنای تعریف خرید معتبر در مراحل بعدی Data Preparation خواهد بود.

</div>


---